In [1]:


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)




In [5]:
ratings = pd.read_csv(
    "data/ratings.dat",
    sep="::",
    engine="python",
    names=["userId", "movieId", "rating", "timestamp"]
)

movies_ml = pd.read_csv(
    "data/movies.dat",
    sep="::",
    engine="python",
    names=["movieId", "title", "genres"],
    encoding="latin-1"
)


In [6]:
movies_ml.head()

,movieId,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [7]:
movies_meta = pd.read_csv(
    "data/movies_metadata.csv",
    low_memory=False
)

# Keep only valid TMDB ids
movies_meta = movies_meta[movies_meta["id"].str.isnumeric()]
movies_meta["id"] = movies_meta["id"].astype(int)


In [8]:
credits = pd.read_csv("data/credits.csv")
keywords = pd.read_csv("data/keywords.csv")

tmdb = movies_meta.merge(credits, on="id") \
                   .merge(keywords, on="id")


In [9]:
tmdb.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count', 'cast', 'crew', 'keywords'],
      dtype='object')

In [10]:
tmdb["homepage"]

0        http://toystory.disney.com/toy-story
1                                         NaN
2                                         NaN
3                                         NaN
4                                         NaN
                         ...                 
46623    http://www.imdb.com/title/tt6209470/
46624                                     NaN
46625                                     NaN
46626                                     NaN
46627                                     NaN
Name: homepage, Length: 46628, dtype: object

In [11]:
tmdb["poster_path"]

0        /rhIRbceoE9lR4veEXuwCC2wARtG.jpg
1        /vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg
2        /6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg
3        /16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg
4        /e64sOI48hQXyru7naBFyssKFxVd.jpg
                       ...               
46623    /jldsYflnId4tTWPx8es3uzsB1I8.jpg
46624    /xZkmxsNmYXJbKVsTRLLx3pqGHx7.jpg
46625    /d5bX92nDsISNhu3ZT69uHwmfCGw.jpg
46626    /aorBPO7ak8e8iJKT5OcqYxU3jlK.jpg
46627    /s5UkZt6NTsrS7ZF0Rh8nzupRlIU.jpg
Name: poster_path, Length: 46628, dtype: object

In [13]:
links = pd.read_csv("data/links.csv")

# Clean ids
links = links.dropna(subset=["tmdbId", "movieId"])
links["tmdbId"] = links["tmdbId"].astype(int)


In [14]:
movies_common = movies_ml.merge(
    links[["movieId", "tmdbId"]],
    on="movieId",
    how="inner"
).merge(
    tmdb,
    left_on="tmdbId",
    right_on="id",
    how="inner"
)


In [15]:
print("MovieLens movies:", movies_ml.movieId.nunique())
print("Common movies:", movies_common.movieId.nunique())


MovieLens movies: 3883
Common movies: 3828


In [16]:
ratings.shape

(1000209, 4)

In [17]:
common_movie_ids = set(movies_common.movieId)

ratings_clean = ratings[
    ratings.movieId.isin(common_movie_ids)
]


In [18]:
ratings.shape

(1000209, 4)

In [19]:
ratings_clean.shape


(997407, 4)

In [20]:
ratings_clean

,userId,movieId,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,5,956704887
1000206,6040,562,5,956704746
1000207,6040,1096,4,956715648


In [21]:
movies_common.isnull().sum()

movieId                     0
title_x                     0
genres_x                    0
tmdbId                      0
adult                       0
belongs_to_collection    3194
budget                      0
genres_y                    0
homepage                 3623
id                          0
imdb_id                     0
original_language           0
original_title              0
overview                   18
popularity                  0
poster_path                18
production_companies        0
production_countries        0
release_date                4
revenue                     0
runtime                     6
spoken_languages            0
status                      5
tagline                   941
title_y                     0
video                       0
vote_average                0
vote_count                  0
cast                        0
crew                        0
keywords                    0
dtype: int64

In [22]:
content_df = movies_common[[
    "movieId",
    "title_x",
    "overview",
    "genres_x",
    "keywords",
    "cast",
    "crew"
]].copy()


In [23]:
content_df.isnull().sum()

movieId      0
title_x      0
overview    18
genres_x     0
keywords     0
cast         0
crew         0
dtype: int64

In [24]:
content_df.rename(columns={"title_x":"title","genres_x":"genres"},inplace=True)

In [25]:
content_df.dropna(inplace=True)

In [26]:
content_df

,movieId,title,overview,genres,keywords,cast,crew
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation|Children's|Comedy,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...","[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure|Children's|Fantasy,"[{'id': 10090, 'name': 'board game'}, {'id': 1...","[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy|Romance,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392...","[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de..."
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy|Drama,"[{'id': 818, 'name': 'based on novel'}, {'id':...","[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de..."
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n...","[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de..."
...,...,...,...,...,...,...,...
3858,3948,Meet the Parents (2000),"Greg Focker is ready to marry his girlfriend, ...",Comedy,"[{'id': 591, 'name': 'cia'}, {'id': 822, 'name...","[{'cast_id': 1, 'character': 'Gaylord ""Greg"" F...","[{'credit_id': '52fe4303c3a36847f8033ca9', 'de..."
3859,3949,Requiem for a Dream (2000),The hopes and dreams of four ambitious people ...,Drama,"[{'id': 1803, 'name': 'drug addiction'}, {'id'...","[{'cast_id': 29, 'character': 'Sara Goldfarb',...","[{'credit_id': '52fe4263c3a36847f801a72b', 'de..."
3860,3950,Tigerland (2000),A group of recruits go through Advanced Infant...,Drama,"[{'id': 10183, 'name': 'independent film'}, {'...","[{'cast_id': 1, 'character': 'Pvt. Roland Bozz...","[{'credit_id': '52fe43a39251416c75018317', 'de..."
3861,3951,Two Family House (2000),Buddy Visalo (Michael Rispoli) is a factory wo...,Drama,[],"[{'cast_id': 1, 'character': 'Buddy Visalo', '...","[{'credit_id': '52fe46c2c3a368484e0a2471', 'de..."


In [27]:
content_df["genres"]=content_df["genres"].str.replace("|"," ")

C:\Users\Sinan\AppData\Local\Temp\ipykernel_13828\2574255875.py:1: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  content_df["genres"]=content_df["genres"].str.replace("|"," ")


In [28]:
content_df

,movieId,title,overview,genres,keywords,cast,crew
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Children's Comedy,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...","[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Children's Fantasy,"[{'id': 10090, 'name': 'board game'}, {'id': 1...","[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy Romance,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392...","[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de..."
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama,"[{'id': 818, 'name': 'based on novel'}, {'id':...","[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de..."
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n...","[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de..."
...,...,...,...,...,...,...,...
3858,3948,Meet the Parents (2000),"Greg Focker is ready to marry his girlfriend, ...",Comedy,"[{'id': 591, 'name': 'cia'}, {'id': 822, 'name...","[{'cast_id': 1, 'character': 'Gaylord ""Greg"" F...","[{'credit_id': '52fe4303c3a36847f8033ca9', 'de..."
3859,3949,Requiem for a Dream (2000),The hopes and dreams of four ambitious people ...,Drama,"[{'id': 1803, 'name': 'drug addiction'}, {'id'...","[{'cast_id': 29, 'character': 'Sara Goldfarb',...","[{'credit_id': '52fe4263c3a36847f801a72b', 'de..."
3860,3950,Tigerland (2000),A group of recruits go through Advanced Infant...,Drama,"[{'id': 10183, 'name': 'independent film'}, {'...","[{'cast_id': 1, 'character': 'Pvt. Roland Bozz...","[{'credit_id': '52fe43a39251416c75018317', 'de..."
3861,3951,Two Family House (2000),Buddy Visalo (Michael Rispoli) is a factory wo...,Drama,[],"[{'cast_id': 1, 'character': 'Buddy Visalo', '...","[{'credit_id': '52fe46c2c3a368484e0a2471', 'de..."


In [29]:
import ast


In [30]:
def extract_names(text):
    data=ast.literal_eval(text)
    return " ".join([d["name"] for d in data])
    

In [31]:
content_df["keywords"][0]

"[{'id': 931, 'name': 'jealousy'}, {'id': 4290, 'name': 'toy'}, {'id': 5202, 'name': 'boy'}, {'id': 6054, 'name': 'friendship'}, {'id': 9713, 'name': 'friends'}, {'id': 9823, 'name': 'rivalry'}, {'id': 165503, 'name': 'boy next door'}, {'id': 170722, 'name': 'new toy'}, {'id': 187065, 'name': 'toy comes to life'}]"

In [32]:
extract_names(content_df["keywords"][0])

'jealousy toy boy friendship friends rivalry boy next door new toy toy comes to life'

In [33]:
content_df["keywords"] = content_df["keywords"].apply(extract_names)
# content_df["cast"] = content_df["cast"].apply(extract_names)
# content_df["crew"] = content_df["crew"].apply(extract_names)


In [34]:
content_df

,movieId,title,overview,genres,keywords,cast,crew
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Children's Comedy,jealousy toy boy friendship friends rivalry bo...,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Children's Fantasy,board game disappearance based on children's b...,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy Romance,fishing best friend duringcreditsstinger old men,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de..."
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama,based on novel interracial relationship single...,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de..."
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,baby midlife crisis confidence aging daughter ...,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de..."
...,...,...,...,...,...,...,...
3858,3948,Meet the Parents (2000),"Greg Focker is ready to marry his girlfriend, ...",Comedy,cia airport cat jew orderly airplane father-in...,"[{'cast_id': 1, 'character': 'Gaylord ""Greg"" F...","[{'credit_id': '52fe4303c3a36847f8033ca9', 'de..."
3859,3949,Requiem for a Dream (2000),The hopes and dreams of four ambitious people ...,Drama,drug addiction junkie heroin speed diet unsoci...,"[{'cast_id': 29, 'character': 'Sara Goldfarb',...","[{'credit_id': '52fe4263c3a36847f801a72b', 'de..."
3860,3950,Tigerland (2000),A group of recruits go through Advanced Infant...,Drama,independent film awol fort polk louisiana targ...,"[{'cast_id': 1, 'character': 'Pvt. Roland Bozz...","[{'credit_id': '52fe43a39251416c75018317', 'de..."
3861,3951,Two Family House (2000),Buddy Visalo (Michael Rispoli) is a factory wo...,Drama,,"[{'cast_id': 1, 'character': 'Buddy Visalo', '...","[{'credit_id': '52fe46c2c3a368484e0a2471', 'de..."


In [35]:
content_df["cast"][0]

"[{'cast_id': 14, 'character': 'Woody (voice)', 'credit_id': '52fe4284c3a36847f8024f95', 'gender': 2, 'id': 31, 'name': 'Tom Hanks', 'order': 0, 'profile_path': '/pQFoyx7rp09CJTAb932F2g8Nlho.jpg'}, {'cast_id': 15, 'character': 'Buzz Lightyear (voice)', 'credit_id': '52fe4284c3a36847f8024f99', 'gender': 2, 'id': 12898, 'name': 'Tim Allen', 'order': 1, 'profile_path': '/uX2xVf6pMmPepxnvFWyBtjexzgY.jpg'}, {'cast_id': 16, 'character': 'Mr. Potato Head (voice)', 'credit_id': '52fe4284c3a36847f8024f9d', 'gender': 2, 'id': 7167, 'name': 'Don Rickles', 'order': 2, 'profile_path': '/h5BcaDMPRVLHLDzbQavec4xfSdt.jpg'}, {'cast_id': 17, 'character': 'Slinky Dog (voice)', 'credit_id': '52fe4284c3a36847f8024fa1', 'gender': 2, 'id': 12899, 'name': 'Jim Varney', 'order': 3, 'profile_path': '/eIo2jVVXYgjDtaHoF19Ll9vtW7h.jpg'}, {'cast_id': 18, 'character': 'Rex (voice)', 'credit_id': '52fe4284c3a36847f8024fa5', 'gender': 2, 'id': 12900, 'name': 'Wallace Shawn', 'order': 4, 'profile_path': '/oGE6JqPP2xH4t

In [36]:
def convert_cast(text):
    return " ".join([d["name"] for d in ast.literal_eval(text)][:3])

In [37]:
content_df["cast"] = content_df["cast"].apply(convert_cast)

In [38]:
content_df.head()

,movieId,title,overview,genres,keywords,cast,crew
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Children's Comedy,jealousy toy boy friendship friends rivalry bo...,Tom Hanks Tim Allen Don Rickles,"[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Children's Fantasy,board game disappearance based on children's b...,Robin Williams Jonathan Hyde Kirsten Dunst,"[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy Romance,fishing best friend duringcreditsstinger old men,Walter Matthau Jack Lemmon Ann-Margret,"[{'credit_id': '52fe466a9251416c75077a89', 'de..."
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama,based on novel interracial relationship single...,Whitney Houston Angela Bassett Loretta Devine,"[{'credit_id': '52fe44779251416c91011acb', 'de..."
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,baby midlife crisis confidence aging daughter ...,Steve Martin Diane Keaton Martin Short,"[{'credit_id': '52fe44959251416c75039ed7', 'de..."


In [39]:
content_df["crew"][0]

'[{\'credit_id\': \'52fe4284c3a36847f8024f49\', \'department\': \'Directing\', \'gender\': 2, \'id\': 7879, \'job\': \'Director\', \'name\': \'John Lasseter\', \'profile_path\': \'/7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg\'}, {\'credit_id\': \'52fe4284c3a36847f8024f4f\', \'department\': \'Writing\', \'gender\': 2, \'id\': 12891, \'job\': \'Screenplay\', \'name\': \'Joss Whedon\', \'profile_path\': \'/dTiVsuaTVTeGmvkhcyJvKp2A5kr.jpg\'}, {\'credit_id\': \'52fe4284c3a36847f8024f55\', \'department\': \'Writing\', \'gender\': 2, \'id\': 7, \'job\': \'Screenplay\', \'name\': \'Andrew Stanton\', \'profile_path\': \'/pvQWsu0qc8JFQhMVJkTHuexUAa1.jpg\'}, {\'credit_id\': \'52fe4284c3a36847f8024f5b\', \'department\': \'Writing\', \'gender\': 2, \'id\': 12892, \'job\': \'Screenplay\', \'name\': \'Joel Cohen\', \'profile_path\': \'/dAubAiZcvKFbboWlj7oXOkZnTSu.jpg\'}, {\'credit_id\': \'52fe4284c3a36847f8024f61\', \'department\': \'Writing\', \'gender\': 0, \'id\': 12893, \'job\': \'Screenplay\', \'name\': \'A

In [40]:
def convert_crew(text):
    return " ".join([d["name"] for d in ast.literal_eval(text) if d['job']=="Director"])

In [41]:
content_df["crew"] = content_df["crew"].apply(convert_crew)

In [42]:
content_df.head()

,movieId,title,overview,genres,keywords,cast,crew
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Children's Comedy,jealousy toy boy friendship friends rivalry bo...,Tom Hanks Tim Allen Don Rickles,John Lasseter
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Children's Fantasy,board game disappearance based on children's b...,Robin Williams Jonathan Hyde Kirsten Dunst,Joe Johnston
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy Romance,fishing best friend duringcreditsstinger old men,Walter Matthau Jack Lemmon Ann-Margret,Howard Deutch
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama,based on novel interracial relationship single...,Whitney Houston Angela Bassett Loretta Devine,Forest Whitaker
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,baby midlife crisis confidence aging daughter ...,Steve Martin Diane Keaton Martin Short,Charles Shyer


In [43]:
content_df["combined_features"] = (
    # content_df["overview"] + " " +
    (content_df["genres"] + " ") * 3 + 
    content_df["keywords"] + " " +
    content_df["cast"]+" "+content_df["crew"]
)


In [44]:
# Clean genres
content_df["genres_clean"] = (
    content_df["genres"]
    .str.replace("|", " ", regex=False)
    .str.lower()
)

# Clean keywords (already space separated usually)
content_df["keywords_clean"] = (
    content_df["keywords"]
    .str.lower()
)

# Final explainable text
content_df["explainable_text"] = (
    content_df["genres_clean"] + " " +
    content_df["keywords_clean"]
)


In [45]:
content_df.head()

,movieId,title,overview,genres,keywords,cast,crew,combined_features,genres_clean,keywords_clean,explainable_text
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Children's Comedy,jealousy toy boy friendship friends rivalry bo...,Tom Hanks Tim Allen Don Rickles,John Lasseter,Animation Children's Comedy Animation Children...,animation children's comedy,jealousy toy boy friendship friends rivalry bo...,animation children's comedy jealousy toy boy f...
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Children's Fantasy,board game disappearance based on children's b...,Robin Williams Jonathan Hyde Kirsten Dunst,Joe Johnston,Adventure Children's Fantasy Adventure Childre...,adventure children's fantasy,board game disappearance based on children's b...,adventure children's fantasy board game disapp...
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy Romance,fishing best friend duringcreditsstinger old men,Walter Matthau Jack Lemmon Ann-Margret,Howard Deutch,Comedy Romance Comedy Romance Comedy Romance f...,comedy romance,fishing best friend duringcreditsstinger old men,comedy romance fishing best friend duringcredi...
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama,based on novel interracial relationship single...,Whitney Houston Angela Bassett Loretta Devine,Forest Whitaker,Comedy Drama Comedy Drama Comedy Drama based o...,comedy drama,based on novel interracial relationship single...,comedy drama based on novel interracial relati...
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,baby midlife crisis confidence aging daughter ...,Steve Martin Diane Keaton Martin Short,Charles Shyer,Comedy Comedy Comedy baby midlife crisis confi...,comedy,baby midlife crisis confidence aging daughter ...,comedy baby midlife crisis confidence aging da...


In [46]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_sim = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

tfidf_matrix_sim = tfidf_sim.fit_transform(
    content_df["combined_features"]
)


In [47]:
cosine_sim = cosine_similarity(
    tfidf_matrix_sim,
    tfidf_matrix_sim
)


In [48]:

import pickle

with open("tfidf.pkl", "wb") as f:
    pickle.dump(tfidf_sim, f)

with open("cosine_sim.pkl", "wb") as f:
    pickle.dump(cosine_sim, f)


with open("svd_model.pkl", "wb") as f:
    pickle.dump(svd, f)


NameError: name 'svd' is not defined

In [ ]:
movie_index = pd.Series(
    content_df.index,
    index=content_df["movieId"]
)


In [49]:
def recommend_movies_content(movie_id, top_n=10):
    if movie_id not in movie_index:
        return "Movie not found"

    idx = movie_index[movie_id]

    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n + 1]

    movie_indices = [i[0] for i in similarity_scores]

    return content_df.iloc[movie_indices][["movieId", "title"]]


In [50]:
content_df[content_df["title"] == "Jumanji (1995)"][["movieId", "title"]]


,movieId,title
1,2,Jumanji (1995)


In [51]:
recommend_movies_content(movie_id=2	)
#jumanji

NameError: name 'movie_index' is not defined

In [52]:
recommend_movies_content(movie_id=1	)
#toy story

NameError: name 'movie_index' is not defined

In [53]:
content_df.sample(10)

,movieId,title,overview,genres,keywords,cast,crew,combined_features,genres_clean,keywords_clean,explainable_text
2661,2741,No Mercy (1986),An unconventional undercover Chicago cop and h...,Action Thriller,sex louisiana vigilante revenge murder violenc...,Richard Gere Kim Basinger Jeroen Krabbé,Richard Pearce,Action Thriller Action Thriller Action Thrille...,action thriller,sex louisiana vigilante revenge murder violenc...,action thriller sex louisiana vigilante reveng...
355,360,I Love Trouble (1994),Rival Chicago reporters Sabrina Peterson (Robe...,Action Comedy,newspaper adversary reporter experience,Nick Nolte Saul Rubinek James Rebhorn,Charles Shyer,Action Comedy Action Comedy Action Comedy news...,action comedy,newspaper adversary reporter experience,action comedy newspaper adversary reporter exp...
185,188,"Prophecy, The (1995)",The angel Gabriel comes to Earth to collect a ...,Horror,angel archangel gabriel menschheit,Christopher Walken Elias Koteas Virginia Madsen,Gregory Widen,Horror Horror Horror angel archangel gabriel m...,horror,angel archangel gabriel menschheit,horror angel archangel gabriel menschheit
1339,1358,Sling Blade (1996),Karl Childers is a mentally disabled man who h...,Drama Thriller,independent film repair shop southern death th...,Billy Bob Thornton Dwight Yoakam J.T. Walsh,Billy Bob Thornton,Drama Thriller Drama Thriller Drama Thriller i...,drama thriller,independent film repair shop southern death th...,drama thriller independent film repair shop so...
349,354,Cobb (1994),Al Stump is a famous sports-writer chosen by T...,Drama,baseball sport historical figure,Tommy Lee Jones Robert Wuhl Lolita Davidovich,Ron Shelton,Drama Drama Drama baseball sport historical fi...,drama,baseball sport historical figure,drama baseball sport historical figure
2865,2946,Help! (1965),"This Beatles film has an obscure Asian cult, t...",Comedy Musical,musical music cult mad scientist the beatles,George Harrison John Lennon Paul McCartney,Richard Lester,Comedy Musical Comedy Musical Comedy Musical m...,comedy musical,musical music cult mad scientist the beatles,comedy musical musical music cult mad scientis...
2020,2101,Squanto: A Warrior's Tale (1994),Squanto is a high-born Indian warrior from a t...,Adventure Drama,,Adam Beach Sheldon Peters Wolfchild Irene Bedard,Xavier Koller,Adventure Drama Adventure Drama Adventure Dram...,adventure drama,,adventure drama
3284,3368,"Big Country, The (1958)","Retired, wealthy sea Captain Jame McKay arrive...",Romance Western,love triangle father son relationship ranch fi...,Gregory Peck Jean Simmons Carroll Baker,William Wyler,Romance Western Romance Western Romance Wester...,romance western,love triangle father son relationship ranch fi...,romance western love triangle father son relat...
1090,1093,"Doors, The (1991)",The story of the famous and influential 1960's...,Drama Musical,hippie poetry sex rock and roll nudity halluci...,Val Kilmer Meg Ryan Kyle MacLachlan,Oliver Stone,Drama Musical Drama Musical Drama Musical hipp...,drama musical,hippie poetry sex rock and roll nudity halluci...,drama musical hippie poetry sex rock and roll ...
1085,1088,Dirty Dancing (1987),Expecting the usual tedium that accompanies a ...,Musical Romance,dancing sex hotel robbery sister sister relati...,Jennifer Grey Patrick Swayze Jerry Orbach,Emile Ardolino,Musical Romance Musical Romance Musical Romanc...,musical romance,dancing sex hotel robbery sister sister relati...,musical romance dancing sex hotel robbery sist...


In [54]:
ratings_clean


,userId,movieId,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,5,956704887
1000206,6040,562,5,956704746
1000207,6040,1096,4,956715648


In [55]:
import pandas as pd
import numpy as np
from collections import defaultdict


In [2]:

import surprise
print(surprise.__version__)


ModuleNotFoundError: No module named 'surprise'

In [62]:
!pip install scikit-surprise


  Using cached scikit_surprise-1.1.4.tar.gz (154 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build scikit-surprise


  error: subprocess-exited-with-error
  
  × Building wheel for scikit-surprise (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [155 lines of output]
      C:\Users\Sinan\AppData\Local\Temp\pip-build-env-45w33gvi\overlay\Lib\site-packages\setuptools\config\_apply_pyprojecttoml.py:82: SetuptoolsDeprecationWarning: `project.license` as a TOML table is deprecated
      !!
      
              ********************************************************************************
              Please use a simple string containing a SPDX expression for `project.license`. You can also use `project.license-files`. (Both options available on setuptools>=77.0.0).
      
              By 2027-Feb-18, you need to update your project and remove deprecated calls
              or your builds will no longer be supported.
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ****************************************

In [3]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split


ModuleNotFoundError: No module named 'surprise'

In [4]:
reader = Reader(rating_scale=(0.5, 5.0))

data = Dataset.load_from_df(
    ratings_clean[["userId", "movieId", "rating"]],
    reader
)


NameError: name 'Reader' is not defined

In [58]:
trainset, _ = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)


In [5]:
svd = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd.fit(trainset)


NameError: name 'SVD' is not defined

In [60]:
def cf_score(user_id, movie_id):
    return svd.predict(user_id, movie_id).est


In [61]:
def rank_movies_cf(user_id, candidate_movie_ids, top_n=10):
    scores = []

    for movie_id in candidate_movie_ids:
        score = cf_score(user_id, movie_id)
        scores.append((movie_id, score))

    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_n]


In [62]:
# Step 1: get candidates from content-based model
content_candidates = recommend_movies_content(movie_id=1, top_n=10)["movieId"]

# Step 2: rank them using CF
ranked = rank_movies_cf(user_id=10, candidate_movie_ids=content_candidates)

# Step 3: show ranked movies
ranked_movies = pd.DataFrame(ranked, columns=["movieId", "cf_score"])
ranked_movies.merge(
    content_df[["movieId", "title"]],
    on="movieId"
)


,movieId,cf_score,title
0,3751,4.867838,Chicken Run (2000)
1,3114,4.349145,Toy Story 2 (1999)
2,2141,4.223761,"American Tail, An (1986)"
3,2355,4.212605,"Bug's Life, A (1998)"
4,596,3.907944,Pinocchio (1940)
5,2142,3.892748,"American Tail: Fievel Goes West, An (1991)"
6,3611,3.662182,Saludos Amigos (1943)
7,1064,3.613853,Aladdin and the King of Thieves (1996)
8,244,3.471190,Gumby: The Movie (1995)
9,2354,3.335601,"Rugrats Movie, The (1998)"


In [63]:
!pip install shap


Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-2.4.2-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached numpy-2.3.5-cp312-cp312-win_amd64.whl.metadata (60 kB)
Using cached numpy-2.3.5-cp312-cp312-win_amd64.whl (12.8 MB)


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.3.5 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.32.1 which is incompatible.


In [66]:
content_df["genres_clean"] = (
    content_df["genres"]
    .str.replace("|", " ", regex=False)
    .str.lower()
)


In [67]:
def get_set(series_value):
    if isinstance(series_value, str):
        return set(series_value.lower().split())
    return set()


In [68]:
def generate_rich_explanation(seed_movie_id, candidate_movie_id):
    # Seed movie data
    seed_row = content_df[content_df["movieId"] == seed_movie_id].iloc[0]
    cand_row = content_df[content_df["movieId"] == candidate_movie_id].iloc[0]

    seed_title = seed_row["title"]

    # Genres
    seed_genres = get_set(seed_row["genres_clean"])
    cand_genres = get_set(cand_row["genres_clean"])
    shared_genres = seed_genres.intersection(cand_genres)

    # Cast
    seed_cast = get_set(seed_row["cast"])
    cand_cast = get_set(cand_row["cast"])
    shared_cast = seed_cast.intersection(cand_cast)

    # Director
    seed_director = seed_row.get("crew", "")
    cand_director = cand_row.get("crew", "")
    same_director = (
        isinstance(seed_director, str)
        and isinstance(cand_director, str)
        and seed_director.lower() == cand_director.lower()
        and seed_director != ""
    )

    reasons = []

    if shared_genres:
        reasons.append(
            "shared genres like " +
            ", ".join([g.title() for g in list(shared_genres)[:2]])
        )

    if shared_cast:
        reasons.append(
            "common cast members such as " +
            ", ".join([c.title() for c in list(shared_cast)[:2]])
        )

    if same_director:
        reasons.append(
            f"the same director ({seed_director})"
        )

    if not reasons:
        return (
            f"Recommended because it is similar in overall theme to "
            f"'{seed_title}'."
        )

    return (
        f"Recommended because it shares "
        + ", and ".join(reasons)
        + f" with '{seed_title}'."
    )


In [76]:
seed_movie_id = 2

content_candidates = recommend_movies_content(
    movie_id=seed_movie_id,
    top_n=10
)

explanations = {}

for movie_id in content_candidates["movieId"]:
    explanations[movie_id] = generate_rich_explanation(
        seed_movie_id,
        movie_id
    )


In [77]:
ranked = rank_movies_cf(
    user_id=10,
    candidate_movie_ids=content_candidates["movieId"].tolist()
)

ranked_df = (
    pd.DataFrame(ranked, columns=["movieId", "cf_score"])
    .merge(content_df[["movieId", "title"]], on="movieId")
)


In [78]:
ranked_df["explanation"] = ranked_df["movieId"].apply(
    lambda x: explanations[x]
)

ranked_df


,movieId,cf_score,title,explanation
0,2161,4.883018,"NeverEnding Story, The (1984)",Recommended because it shares shared genres li...
1,1967,4.874100,Labyrinth (1986),Recommended because it shares shared genres li...
2,2005,4.333994,"Goonies, The (1985)",Recommended because it shares shared genres li...
3,2043,4.225967,Darby O'Gill and the Little People (1959),Recommended because it shares shared genres li...
4,1009,4.118782,Escape to Witch Mountain (1975),Recommended because it shares shared genres li...
5,60,4.004665,"Indian in the Cupboard, The (1995)",Recommended because it shares shared genres li...
6,2399,3.678919,Santa Claus: The Movie (1985),Recommended because it shares shared genres li...
7,56,3.299471,Kids of the Round Table (1995),Recommended because it shares shared genres li...
8,2162,2.851041,"NeverEnding Story II: The Next Chapter, The (1...",Recommended because it shares shared genres li...
9,126,2.727981,"NeverEnding Story III, The (1994)",Recommended because it shares shared genres li...


In [80]:
ranked_df["explanation"][8]

"Recommended because it shares shared genres like Fantasy, Adventure, and common cast members such as Jonathan with 'Jumanji (1995)'."

In [73]:
content_df

,movieId,title,overview,genres,keywords,cast,crew,combined_features,genres_clean,keywords_clean,explainable_text
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Children's Comedy,jealousy toy boy friendship friends rivalry bo...,Tom Hanks Tim Allen Don Rickles,John Lasseter,Animation Children's Comedy Animation Children...,animation children's comedy,jealousy toy boy friendship friends rivalry bo...,animation children's comedy jealousy toy boy f...
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Children's Fantasy,board game disappearance based on children's b...,Robin Williams Jonathan Hyde Kirsten Dunst,Joe Johnston,Adventure Children's Fantasy Adventure Childre...,adventure children's fantasy,board game disappearance based on children's b...,adventure children's fantasy board game disapp...
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy Romance,fishing best friend duringcreditsstinger old men,Walter Matthau Jack Lemmon Ann-Margret,Howard Deutch,Comedy Romance Comedy Romance Comedy Romance f...,comedy romance,fishing best friend duringcreditsstinger old men,comedy romance fishing best friend duringcredi...
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama,based on novel interracial relationship single...,Whitney Houston Angela Bassett Loretta Devine,Forest Whitaker,Comedy Drama Comedy Drama Comedy Drama based o...,comedy drama,based on novel interracial relationship single...,comedy drama based on novel interracial relati...
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,baby midlife crisis confidence aging daughter ...,Steve Martin Diane Keaton Martin Short,Charles Shyer,Comedy Comedy Comedy baby midlife crisis confi...,comedy,baby midlife crisis confidence aging daughter ...,comedy baby midlife crisis confidence aging da...
...,...,...,...,...,...,...,...,...,...,...,...
3858,3948,Meet the Parents (2000),"Greg Focker is ready to marry his girlfriend, ...",Comedy,cia airport cat jew orderly airplane father-in...,Ben Stiller Robert De Niro Teri Polo,Jay Roach,Comedy Comedy Comedy cia airport cat jew order...,comedy,cia airport cat jew orderly airplane father-in...,comedy cia airport cat jew orderly airplane fa...
3859,3949,Requiem for a Dream (2000),The hopes and dreams of four ambitious people ...,Drama,drug addiction junkie heroin speed diet unsoci...,Ellen Burstyn Jared Leto Jennifer Connelly,Darren Aronofsky,Drama Drama Drama drug addiction junkie heroin...,drama,drug addiction junkie heroin speed diet unsoci...,drama drug addiction junkie heroin speed diet ...
3860,3950,Tigerland (2000),A group of recruits go through Advanced Infant...,Drama,independent film awol fort polk louisiana targ...,Colin Farrell Matthew Davis Clifton Collins Jr,Joel Schumacher,Drama Drama Drama independent film awol fort p...,drama,independent film awol fort polk louisiana targ...,drama independent film awol fort polk louisian...
3861,3951,Two Family House (2000),Buddy Visalo (Michael Rispoli) is a factory wo...,Drama,,Michael Rispoli Kelly Macdonald Kathrine Narducci,Raymond De Felitta,Drama Drama Drama Michael Rispoli Kelly Macdo...,drama,,drama


In [104]:
import os

os.makedirs("models", exist_ok=True)


In [105]:
import pickle

with open("models/tfidf.pkl", "wb") as f:
    pickle.dump(tfidf_sim, f)

with open("models/cosine_sim.pkl", "wb") as f:
    pickle.dump(cosine_sim, f)

with open("models/svd_model.pkl", "wb") as f:
    pickle.dump(svd, f)
